# 08. 이미지 필터링

평균·가우시안·샤프닝·메디안·양방향 필터와 카툰 카메라를 실습합니다.

강의 슬라이드의 코드를 실행 순서에 맞게 정리한 실습 노트북입니다.

> 이미지 예제는 노트북과 같은 위치에 `data` 폴더를 만들고 강의에서 사용하는 파일을 넣어 실행하세요.  
> `cv2.imshow()`와 카메라·마우스 예제는 데스크톱 Jupyter/VS Code 환경에서 실행하는 것을 권장합니다.


In [ ]:
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt

def read_image(name, mode=cv2.IMREAD_COLOR):
    image = cv2.imread(str(Path("data") / name), mode)
    if image is None:
        raise FileNotFoundError(f"data/{name}")
    return image

def show(images, titles):
    fig, axes = plt.subplots(1, len(images), figsize=(5 * len(images), 4))
    axes = np.atleast_1d(axes)
    for ax, image, title in zip(axes, images, titles):
        ax.imshow(image if image.ndim == 2 else cv2.cvtColor(image, cv2.COLOR_BGR2RGB),
                  cmap="gray" if image.ndim == 2 else None)
        ax.set_title(title); ax.axis("off")
    plt.show()


## 평균값 필터


In [ ]:
src = read_image("rose.bmp", cv2.IMREAD_GRAYSCALE)
kernel = np.ones((5, 5), dtype=np.float32) / 25
filtered = cv2.filter2D(src, -1, kernel)
blurred = cv2.blur(src, (5, 5))
show([src, filtered, blurred], ["source", "filter2D", "blur"])

sizes = [3, 5, 7]
results = [cv2.blur(src, (size, size)) for size in sizes]
show(results, [f"mean {size}x{size}" for size in sizes])


## 가우시안 필터와 언샤프 마스크


In [ ]:
gaussians = [cv2.GaussianBlur(src, (0, 0), sigma) for sigma in range(1, 6)]
show(gaussians, [f"sigma={sigma}" for sigma in range(1, 6)])

smooth = cv2.GaussianBlur(src, (0, 0), 2)
sharp = np.clip(2.0 * src - smooth, 0, 255).astype(np.uint8)
show([src, smooth, sharp], ["source", "gaussian", "sharp"])


## 메디안·양방향 필터


In [ ]:
noise = read_image("noise.bmp", cv2.IMREAD_GRAYSCALE)
median = cv2.medianBlur(noise, 3)
show([noise, median], ["noise", "median"])

color = read_image("lenna.bmp")
bilateral = cv2.bilateralFilter(color, -1, 10, 5)
show([color, bilateral], ["source", "bilateral"])


## 카툰·연필 스케치 필터


In [ ]:
def cartoon_filter(image):
    h, w = image.shape[:2]
    small = cv2.resize(image, (w // 2, h // 2))
    smooth = cv2.bilateralFilter(small, -1, 20, 7)
    edge = 255 - cv2.Canny(small, 80, 120)
    edge = cv2.cvtColor(edge, cv2.COLOR_GRAY2BGR)
    result = cv2.bitwise_and(smooth, edge)
    return cv2.resize(result, (w, h), interpolation=cv2.INTER_NEAREST)

def pencil_sketch(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (0, 0), 3)
    return cv2.divide(gray, blur, scale=255)

sample = read_image("lenna.bmp")
show([sample, cartoon_filter(sample), pencil_sketch(sample)],
     ["source", "cartoon", "pencil"])


## 실시간 필터 카메라

Space로 일반·카툰·연필 모드를 바꾸고 ESC로 종료합니다.


In [ ]:
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    raise RuntimeError("카메라를 열 수 없습니다.")

mode = 0
while True:
    ok, frame = cap.read()
    if not ok:
        break
    if mode == 1:
        frame = cartoon_filter(frame)
    elif mode == 2:
        frame = cv2.cvtColor(pencil_sketch(frame), cv2.COLOR_GRAY2BGR)
    cv2.imshow("filter camera", frame)
    key = cv2.waitKey(1) & 0xFF
    if key == 27:
        break
    if key == ord(" "):
        mode = (mode + 1) % 3

cap.release()
cv2.destroyAllWindows()
